In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def find_project_dir():
    override = os.getenv("PLANT_DISEASE_PROJECT_DIR")
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "src" / "data_utils.py").is_file():
            return candidate
        raise FileNotFoundError(f"Thư mục dự án không hợp lệ: {candidate}")

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, cwd / "plant-disease-classification"]
    for candidate in candidates:
        if (candidate / "src" / "data_utils.py").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy dự án. Hãy đặt PLANT_DISEASE_PROJECT_DIR.")

project_dir = find_project_dir()
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

sys.dont_write_bytecode = True

from src.data_utils import (
    DATA_DIR,
    TRAIN_DIR,
    VAL_DIR,
    METADATA_DIR,
    find_exact_duplicates,
    find_near_duplicates,
)


# I. DATA STRUCTURE

# 1 .Xem DATASET có gì

In [ ]:
print(DATA_DIR)
print(TRAIN_DIR.exists())
print(VAL_DIR.exists())

# 2. Lấy danh sách class và đếm số class từng file

In [ ]:
train_classes = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
val_classes = sorted([p.name for p in VAL_DIR.iterdir() if p.is_dir()])

print("Số class train:", len(train_classes))
print("Số class val:", len(val_classes))
if train_classes != val_classes:
    raise ValueError("Danh sách lớp train và val gốc không khớp.")

print("\nMột vài class train:")
print(train_classes[:10])

Danh sách lớp ở `train` và `val` gốc phải giống nhau. Ô phía trên dừng chạy nếu thiếu lớp ở một trong hai tập.

# 3. Đếm ảnh và tạo bảng số lượng từng class

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def count_images(folder):
    return sum(
        1
        for class_dir in folder.iterdir() if class_dir.is_dir()
        for path in class_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    )

n_train = count_images(TRAIN_DIR)
n_val = count_images(VAL_DIR)

print("Train images:", n_train)
print("Val images:", n_val)
print("Total:", n_train + n_val)

In [ ]:
total = n_train + n_val

print(f"Train: {n_train / total:.2%}")
print(f"Val:   {n_val / total:.2%}")

In [ ]:
def count_images_per_class(split_dir):
    result = {}

    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue

        count = sum(
            1
            for path in class_dir.iterdir()
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTS
        )

        result[class_dir.name] = count

    return result

In [ ]:
train_counts = count_images_per_class(TRAIN_DIR)
val_counts = count_images_per_class(VAL_DIR)

split_summary = pd.DataFrame({
    "class_name": train_classes,
    "train_count": [
        train_counts.get(cls, 0)
        for cls in train_classes
    ],
    "val_count": [
        val_counts.get(cls, 0)
        for cls in train_classes
    ]
})

split_summary["total"] = (
    split_summary["train_count"]
    + split_summary["val_count"]
)

split_summary.head()

# II. TẠO BẢNG METADATA

In [ ]:
records = []

for split in ["train", "val"]:

    split_dir = DATA_DIR / split

    for class_dir in split_dir.iterdir():

        if not class_dir.is_dir():
            continue

        class_name = class_dir.name

        plant, condition = class_name.split("___", 1)

        for img_path in class_dir.iterdir():

            if (
                img_path.is_file()
                and img_path.suffix.lower() in IMAGE_EXTS
            ):

                records.append({
                    "relative_path": img_path.relative_to(DATA_DIR).as_posix(),
                    "split": split,
                    "class_name": class_name,
                    "plant": plant,
                    "condition": condition,
                    "is_healthy":
                        condition.lower() == "healthy"
                })

df = pd.DataFrame(records)

In [ ]:
METADATA_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(
    METADATA_DIR / "metadata.csv",
    index=False
)

# III. Thống kê tổng quan DATASET

In [ ]:
total_images = len(df)

train_images = (df["split"] == "train").sum()
val_images = (df["split"] == "val").sum()

num_classes = df["class_name"].nunique()
num_plants = df["plant"].nunique()

healthy_images = df["is_healthy"].sum()
diseased_images = (~df["is_healthy"]).sum()

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "Total images",
        "Train images",
        "Original val images",
        "Number of classes",
        "Number of plant types",
        "Healthy images",
        "Diseased images"
    ],
    "value": [
        total_images,
        train_images,
        val_images,
        num_classes,
        num_plants,
        healthy_images,
        diseased_images
    ]
})

summary

In [ ]:
# Tỉ lệ healthy/diseased
print(f"Healthy ratio: {healthy_images / total_images:.2%}")
print(f"Diseased ratio: {diseased_images / total_images:.2%}")

In [ ]:
# Số loại cây

plant_counts = df["plant"].value_counts()

plant_counts

# IV. Phân bố lớp

In [ ]:
# Số ảnh mỗi class

class_distribution = (
    df["class_name"]
    .value_counts()
    .reset_index()
)

class_distribution.columns = [
    "class_name",
    "count"
]

class_distribution

In [ ]:
class_counts = (
    df["class_name"]
    .value_counts()
)

plt.figure(figsize=(12, 12))

class_counts.sort_values(
    ascending=True
).plot(
    kind="barh"
)

plt.xlabel("Number of images")
plt.ylabel("Class")
plt.title("Class Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# So sánh giữa class lớn nhất và nhỏ nhất 

max_count = class_counts.max()
min_count = class_counts.min()

imbalance_ratio = max_count / min_count

print("Largest class:", class_counts.idxmax())
print("Largest class count:", max_count)

print("Smallest class:", class_counts.idxmin())
print("Smallest class count:", min_count)

print(f"Imbalance ratio: {imbalance_ratio:.2f}")

In [ ]:
class_counts.describe()

In [ ]:
split_class_counts = (
    df.groupby(["class_name", "split"])
    .size()
    .unstack(fill_value=0)
)

# So sánh train và val theo từng class

split_class_counts["total"] = (
    split_class_counts.get("train", 0)
    + split_class_counts.get("val", 0)
)

split_class_counts

In [ ]:
# Tính tỉ lệ val gốc

split_class_counts["val_ratio"] = (
    split_class_counts["val"]
    / split_class_counts["total"]
)

split_class_counts[
    ["train", "val", "total", "val_ratio"]
]

- Tỉ lệ `val` gốc giữa các lớp khá đều; preprocessing sẽ dùng tập này làm **test** và tạo validation mới từ `train` gốc.

In [ ]:
plot_df = (
    df.groupby(
        ["class_name", "split"]
    )
    .size()
    .unstack(fill_value=0)
)

plot_df.plot(
    kind="bar",
    figsize=(16, 7)
)

plt.xlabel("Class")
plt.ylabel("Number of images")
plt.title("Train vs Original Val Distribution per Class")

plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
plant_counts = (
    df["plant"]
    .value_counts()
    .sort_values()
)

plt.figure(figsize=(10, 6))

plant_counts.plot(
    kind="barh"
)

plt.xlabel("Number of images")
plt.ylabel("Plant")
plt.title("Image Distribution by Plant Type")

plt.tight_layout()
plt.show()

# Kết luận phân bố lớp

Các số liệu về lớp lớn nhất, lớp nhỏ nhất và tỉ lệ chênh lệch được tính ở ô phía trên. Tập `val` ở dữ liệu gốc được bước preprocessing dùng làm **test**; validation mới được chia từ `train` gốc.


# V. Hiển thị ảnh mẫu

In [ ]:
# Tạo dataframe train

train_df = df[df["split"] == "train"].copy()

In [ ]:
# Hiển thị ảnh

def show_samples(df, class_name, n=5):

    samples = df[
        df["class_name"] == class_name
    ].sample(
        n=min(
            n,
            len(df[df["class_name"] == class_name])
        ),
        random_state=42
    )

    plt.figure(figsize=(15, 3))

    for i, (_, row) in enumerate(samples.iterrows()):

        img_path = DATA_DIR / row["relative_path"]

        with Image.open(img_path) as img:
            rgb_image = img.convert("RGB")

        plt.subplot(1, len(samples), i + 1)
        plt.imshow(rgb_image)
        plt.axis("off")

    plt.suptitle(class_name)
    plt.tight_layout()
    plt.show()

In [ ]:
# Ví dụ

show_samples(
    train_df,
    "Tomato___Late_blight",
    n=5
)

In [ ]:
# Không nhớ tên class

print(
    df["class_name"]
    .sort_values()
    .unique()
)

In [ ]:
# Xem nhiều class liên tiếp

classes_to_show = [
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___healthy"
]

for cls in classes_to_show:
    show_samples(
        train_df,
        cls,
        n=5
    )

# VI. Kiểm tra kích thước ảnh và kênh màu

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def read_image_info(relative_path):
    try:
        with Image.open(DATA_DIR / relative_path) as img:
            return img.width, img.height, img.mode
    except Exception:
        return None, None, None

with ThreadPoolExecutor(max_workers=16) as pool:
    image_info = list(pool.map(read_image_info, df["relative_path"]))

df[["width", "height", "mode"]] = pd.DataFrame(image_info, index=df.index)
df[["relative_path", "width", "height", "mode"]].head()

In [ ]:
# Thống kê kích thước 

df[
    ["width", "height"]
].describe()



In [ ]:
# Bao nhiêu loại

size_counts = (
    df.groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
)

print(size_counts.to_string())

In [ ]:
# Kiểm tra mode màu
print(
    df["mode"]
    .value_counts(dropna=False)
    .to_string()
)

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    df["width"],
    df["height"],
    alpha=0.3
)

plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Image Size Distribution")

plt.tight_layout()
plt.show()

# Tổng kết kích thước và mode màu

Bảng thống kê và biểu đồ phía trên cho biết kích thước, mode màu thực tế của dữ liệu. Preprocessing sẽ chuyển từng ảnh sang RGB và resize khi đọc batch.

# VII. Kiểm tra chất lượng ảnh

# 1. Ảnh không đọc được


In [ ]:
def verify_image(relative_path):
    try:
        with Image.open(DATA_DIR / relative_path) as img:
            img.verify()
    except Exception as e:
        return {"relative_path": relative_path, "error": str(e)}
    return None

with ThreadPoolExecutor(max_workers=16) as pool:
    broken_files = [result for result in pool.map(verify_image, df["relative_path"]) if result is not None]

broken_df = pd.DataFrame(broken_files, columns=["relative_path", "error"])
print("Broken images:", len(broken_df))
broken_df

# 2. Tìm ảnh trùng hoàn toàn

In [ ]:
df, duplicate_df = find_exact_duplicates(df, DATA_DIR)

In [ ]:
print("Duplicate image rows:", len(duplicate_df))

duplicate_df[
    ["relative_path", "split", "class_name", "sha256"]
]

In [ ]:
duplicate_df[["relative_path", "split", "class_name", "sha256"]].to_csv(
    METADATA_DIR / "duplicate_images.csv",
    index=False
)

# 3. Kiểm tra duplicate giữa train và val

In [ ]:
duplicate_cross_split = (
    duplicate_df
    .groupby("sha256")["split"]
    .nunique()
)

cross_split_hashes = duplicate_cross_split[
    duplicate_cross_split > 1
].index

In [ ]:
cross_split_duplicates = duplicate_df[
    duplicate_df["sha256"].isin(
        cross_split_hashes
    )
]

cross_split_duplicates[
    ["relative_path", "split", "class_name", "sha256"]
]

In [ ]:
print(
    "Duplicate groups across train/val:",
    len(cross_split_hashes)
)

# 4. Kiểm tra cùng ảnh nhưng khác label

In [ ]:
label_counts_per_hash = (
    duplicate_df
    .groupby("sha256")["class_name"]
    .nunique()
)

suspicious_hashes = label_counts_per_hash[
    label_counts_per_hash > 1
].index

In [ ]:
label_conflict_df = duplicate_df[
    duplicate_df["sha256"].isin(
        suspicious_hashes
    )
]

label_conflict_df[
    ["relative_path", "class_name", "sha256"]
]

# 5. Tìm ảnh gần trùng

In [ ]:
near_duplicate_df = find_near_duplicates(df, DATA_DIR, threshold=5)

In [ ]:
print(
    "Near duplicate pairs:",
    len(near_duplicate_df)
)

near_duplicate_df.head()

In [ ]:
near_duplicate_df.to_csv(
    METADATA_DIR / "near_duplicates.csv",
    index=False
)

# Kết luận kiểm tra chất lượng

Các ô phía trên báo cáo số ảnh lỗi, số **dòng ảnh** thuộc nhóm trùng SHA256, số nhóm trùng giữa train/val và số **cặp ảnh** gần trùng theo pHash. `near_duplicates.csv` chỉ chứa cặp cùng lớp; bước preprocessing dùng các cặp này để giữ ảnh gần trùng trong cùng một tập. Không xóa file ảnh gốc.
